# Image Transmission Based on Flask

This chapter introduces how to use Flask to create a web application for displaying real-time video from the robot's camera. Due to the cross-platform nature of web applications, users can watch the camera's real-time video on devices such as smartphones, PCs, tablets, etc., through a browser, achieving wireless image transmission functionality.

## What is Flask?

Flask is a lightweight web application framework used to quickly build web applications using Python.

- Lightweight: Flask is a lightweight framework with a relatively small core library, but it offers enough flexibility and extensibility for developers to choose and add the extensions and libraries they need.
- Simple and Easy to Use: Flask is designed to be simple and easy to use. Its API is clear and well-documented, allowing developers to quickly get started and build web applications rapidly.
- Routing System: Flask uses decorators to define URL routes, mapping requests to corresponding handler functions. This makes creating different pages and handling different requests intuitive and straightforward.
- Template Engine: Flask integrates with the Jinja2 template engine, making it easier to build dynamic content within the application. The template engine allows you to embed dynamically generated content in HTML.
- Integrated Development Server: Flask comes with a simple integrated development server for easy development and debugging. However, in production environments, it is recommended to use more powerful web servers such as Gunicorn or uWSGI.
- Plugins and Extensions: Flask supports many plugins and extensions for adding additional functionality, such as database integration, authentication, form handling, etc.
- RESTful Support: Flask provides good support for RESTful-style APIs, making it simple to build and design RESTful APIs.
- WSGI Compatibility: Flask is based on the Web Server Gateway Interface (WSGI), allowing it to run on many web servers that comply with the WSGI standard.
- Active Community: Flask has a large and active community, meaning you can easily find extensive documentation, tutorials, third-party extensions, and support.d support.

## Preparation
Since the product automatically runs the main program on startup, and the main program occupies the camera resources, this tutorial cannot be used in that state. You need to stop the main program or disable its automatic startup before restarting the robot.

It should be noted that because the robot's main program uses multithreading and is configured to start automatically via a service, the usual method of `sudo killall python` often does not work. Therefore, we will introduce a method to disable the main program's automatic startup.

If you have already disabled the automatic startup of the robot's main program, you do not need to perform the following "Stopping the Main Program" section.

### Stopping the Main Program
1. Click the "" icon next to the tab at the top of this page to open a new tab named Launcher.
2. Click Terminal under Other to open the terminal window.
3. In the terminal window, type `bash` and press Enter.
4. You can now use the Bash Shell to control the robot.
5. Enter the command: `systemctl --user stop ugv-app.service`.
6. On the terminal page, after the command has been executed, continue with the remaining part of this tutorial.

## Web Application Example

### Note: The following code block cannot be run in Jupyter Lab

Due to potential conflicts in port usage between Flask application and Jupyter Lab, the following code cannot be run in Jupyter Lab. The code is stored in the 12 folder within the `tutorial_cn` and `tutorial_en` directories. Inside the `12` folder, there is also a folder named `template` for storing web resources. Below are the steps to run the example:

1. Open a terminal using the method described earlier. Make sure the terminal's default path matches the file path on the left. Navigate to the 12 folder by entering `cd 12` in the terminal.
2. Start the Flask web application server using the following command: `source /home/ws/ugv_rpi/ugv-env/bin/activate && python flask_camera.py`.
3. Open a web browser on a device within the same local network (or open a new tab in the browser on the same device) and enter the Raspberry Pi's IP address followed by :5000. For example, if the Raspberry Pi's IP address is `192.168.10.104`, enter `192.168.10.104:5000` in the browser's address bar. Note that : should be an English colon.
4. Use Ctrl + C in the terminal to end the running application.cation.

### Flask Example

In [ ]:
from flask import Flask, render_template, Response, send_from_directory
import threading
import cv_ctrl
import os

# Create a Flask app instance
app = Flask(__name__)

# Create an instance of the OpenCV function class (for video processing)
cvf = cv_ctrl.OpencvFuncs()

@app.route('/')
def index():
    """
    Home route.
    When visiting http://<ip>:5000/, return the index.html page.
    """
    return render_template('index.html')

@app.route('/<path:filename>')
def serve_static(filename):
    """
    Static file route.
    When a file path is accessed directly, return the corresponding file from the templates directory.
    """
    return send_from_directory('templates', filename)

@app.route('/settings/<path:filename>')
def serve_static_settings(filename):
    """
    Static file route for the settings page.
    Returns the required files for the settings page from the templates directory.
    """
    return send_from_directory('templates', filename)
        
if __name__ == '__main__':
    # Start the camera processing thread
    # daemon=True means it will automatically stop when the main program exits
    cam_thread = threading.Thread(target=cvf.frame_process, daemon=True)
    cam_thread.start()

    # Start the Flask web server
    # host='0.0.0.0' means accessible from external network
    # port=5000 means listen on port 5000
    # debug=False disables debug mode
    app.run(host='0.0.0.0', port=5000, debug=False)

### Explanation of Key Parts of the Code

The core idea is to build a web server using Flask while running OpenCV video processing concurrently in a separate thread. Flask handles browser requests, serving the main page and static resources.

The video processing logic, handled by `cvf.frame_process()`, runs continuously in a separate daemon thread, enabling the front-end page and back-end image processing to operate in parallel.

### Web

In [ ]:
<!doctype html>
<html lang="en">
<head>
    <!-- Required meta tags -->
    <meta charset="utf-8">
    <title>Live Video Based on Flask</title>
</head>
<body>
    <div class="video">
        <video id="video" autoplay muted playsinline></video>
    </div>
    <div id="message"></div>
    <script type="module" src="./webrtc_reader.js"></script>
    <script>
    const videoElement = document.getElementById('video');
    const message = document.getElementById('message');
    const setMessage = (str) => {
    if (str !== '') {
        videoElement.controls = false;
    } else {
        videoElement.controls = defaultControls;
    }
    message.innerText = str;
    };

    const parseBoolString = (str, defaultVal) => {
    str = (str || '');

    if (['1', 'yes', 'true'].includes(str.toLowerCase())) {
        return true;
    }
    if (['0', 'no', 'false'].includes(str.toLowerCase())) {
        return false;
    }
    return defaultVal;
    };

    const loadAttributesFromQuery = () => {
    const params = new URLSearchParams(window.location.search);
    videoElement.controls = parseBoolString(params.get('controls'), false);
    videoElement.muted = parseBoolString(params.get('muted'), true);
    videoElement.autoplay = parseBoolString(params.get('autoplay'), true);
    videoElement.playsInline = parseBoolString(params.get('playsinline'), false);
    defaultControls = videoElement.controls;
    };

    let stream_url = `http://${window.location.hostname}:8889/cam/`;
    window.addEventListener('load', () => {
    loadAttributesFromQuery();
    new MediaMTXWebRTCReader({
        url: new URL('whep', stream_url) + window.location.search,
        onError: (err) => {
        setMessage(err);
        },
        onTrack: (evt) => {
        setMessage('');
        videoElement.srcObject = evt.streams[0];
        },
    });
    });  

    </script>
</body>
</html>

### Comments 

#### **1. HTML Structure**

```html
<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <title>Live Video Based on Flask</title>
</head>
<body>
    <div class="video">
        <video id="video" autoplay muted playsinline></video>
    </div>
    <div id="message"></div>
    <script type="module" src="./webrtc_reader.js"></script>
    <script> ... </script>
</body>
</html>
```

* `<video>`: Used to play the video stream. `autoplay` starts playback automatically, `muted` prevents browser blocking, and `playsinline` allows inline playback on mobile devices.
* `<div id="message">`: Displays error messages or status information.
* `webrtc_reader.js`: External JavaScript module defining `MediaMTXWebRTCReader`, which handles WebRTC video streaming.

#### **2. JavaScript Functionality**

The script mainly handles three tasks:

##### **(1) Handling status messages**

```javascript
const setMessage = (str) => {
    if (str !== '') {
        videoElement.controls = false;  // Hide video controls when a message is shown
    } else {
        videoElement.controls = defaultControls; // Restore controls when no message
    }
    message.innerText = str; // Display the message
};
```

* Displays error messages or clears them.
* Controls whether the `<video>` element shows playback controls.

##### **(2) Load video attributes from URL query parameters**

```javascript
const parseBoolString = (str, defaultVal) => {
    str = (str || '');
    if (['1', 'yes', 'true'].includes(str.toLowerCase())) return true;
    if (['0', 'no', 'false'].includes(str.toLowerCase())) return false;
    return defaultVal;
};

const loadAttributesFromQuery = () => {
    const params = new URLSearchParams(window.location.search);
    videoElement.controls = parseBoolString(params.get('controls'), false);
    videoElement.muted = parseBoolString(params.get('muted'), true);
    videoElement.autoplay = parseBoolString(params.get('autoplay'), true);
    videoElement.playsInline = parseBoolString(params.get('playsinline'), false);
    defaultControls = videoElement.controls;
};
```

* Dynamically sets `<video>` attributes based on URL query parameters, e.g.:

  ```
  http://localhost:5000?controls=true&muted=false
  ```

  This allows controlling playback options from the URL.

##### **(3) Initialize WebRTC video stream**

```javascript
let stream_url = `http://${window.location.hostname}:8889/cam/`;
window.addEventListener('load', () => {
    loadAttributesFromQuery();
    new MediaMTXWebRTCReader({
        url: new URL('whep', stream_url) + window.location.search,
        onError: (err) => {
            setMessage(err);
        },
        onTrack: (evt) => {
            setMessage('');
            videoElement.srcObject = evt.streams[0];
        },
    });
});
```

* Constructs the video stream URL, e.g., `http://<server_ip>:8889/cam/`.
* `MediaMTXWebRTCReader` (from `webrtc_reader.js`) initiates the WebRTC connection:

  * `onError`: Displays an error message if the connection fails.
  * `onTrack`: Assigns the received video track to `<video>`'s `srcObject` for playback.

#### **3. Summary of workflow**

1. Page loads → Reads URL parameters to configure video attributes.
2. Calls `MediaMTXWebRTCReader` → Initiates a WebRTC connection to the streaming server (port 8889).
3. On success → Binds the stream to `<video>` for playback.
4. On error → Displays error message and hides video controls.
